# 03 — Ingredient Extraction
Uses spaCy EntityRuler with a custom ingredient dictionary to find ingredient mentions in each post.

In [ ]:
import spacy
import pandas as pd
from tqdm import tqdm

# python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')
df = pd.read_csv('../data/processed/posts_clean.csv')

In [ ]:
# Custom ingredient dictionary (~200 common skincare actives & irritants)
INGREDIENTS = [
    'retinol', 'retinoid', 'tretinoin', 'retinoic acid',
    'niacinamide', 'vitamin c', 'ascorbic acid', 'vitamin e',
    'hyaluronic acid', 'glycerin', 'ceramide',
    'salicylic acid', 'bha', 'glycolic acid', 'aha', 'lactic acid',
    'benzoyl peroxide', 'azelaic acid',
    'fragrance', 'parfum', 'alcohol', 'denatured alcohol',
    'mineral oil', 'petrolatum', 'lanolin',
    'spf', 'sunscreen', 'zinc oxide', 'titanium dioxide',
    'peptide', 'collagen', 'squalane', 'bakuchiol',
    'tranexamic acid', 'kojic acid', 'arbutin',
]

# Add EntityRuler
ruler = nlp.add_pipe('entity_ruler', before='ner')
patterns = [{'label': 'INGREDIENT', 'pattern': ing} for ing in INGREDIENTS]
ruler.add_patterns(patterns)
print(f'Ingredient dictionary: {len(INGREDIENTS)} entries')

In [ ]:
mentions = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting ingredients'):
    doc = nlp(row['body_clean'])
    for sent in doc.sents:
        for ent in sent.ents:
            if ent.label_ == 'INGREDIENT':
                mentions.append({
                    'post_id':   row['post_id'],
                    'ingredient': ent.text,
                    'sentence':  sent.text,
                })

mentions_df = pd.DataFrame(mentions)
print(f'Total mentions: {len(mentions_df)}')
print(mentions_df['ingredient'].value_counts().head(15))

In [ ]:
mentions_df.to_csv('../data/processed/ingredient_mentions_raw.csv', index=False)
print('Saved ingredient_mentions_raw.csv ✓')